### Imports

In [6]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()


True

In [7]:
# First, create the OpenAI client:

# openai_client = OpenAI()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
# client = genai.Client(api_key=GOOGLE_API_KEY)

# Point the OpenAI client at Google's Gemini OpenAI-compatible endpoint
gemini_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [9]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [10]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=gemini_client,
)

In [13]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join the program even if it has already begun. You can start learning and submitting homework.\n\nHowever, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [14]:
vs_index.close()